### RunnablePassthrough & RunnableLambda & RunnableParallel 

Boilerplate code

In [1]:
import langchain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

google_llm = ChatGoogleGenerativeAI(
    temperature=0,
    model="gemini-2.0-flash",
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0,
    model="gpt-4",
    api_key=openai_api_key
)


#### Callable -plain Python function/lambda/class with call , can be executed with ()

A Callable is a native Python concept that represents anything you can trigger like a function. A Runnable is a LangChain protocol that wraps pieces of code so they can talk to each other in an advanced, automated pipeline.

In [3]:
#callable 

def add_one(x):
    return x+1
add_two =lambda x: x+2

# print(add_one(1))
# print(add_two(1))

print(add_one.__call__(1))
print(add_two.__call__(1))


2
3


In [7]:
#runnable : wrapping callable

from langchain_core.runnables import RunnableLambda
runnable=RunnableLambda(add_one)
print(runnable.invoke(1))

2


#### RunnablePassthrough -just echoes the input unchanged


In [10]:
from langchain_core.runnables import RunnablePassthrough

chain=RunnablePassthrough()

print(chain.invoke("Hello World!"))

Hello World!


using .assign() to do some other works

In [12]:
#adding extra data while keeping the original

chain=RunnablePassthrough.assign(
    length=lambda x: len(x["text"]),
    upper=lambda x: x["text"].upper()

)

result=chain.invoke({"text":"Python"})
print(result)

{'text': 'Python', 'length': 6, 'upper': 'PYTHON'}


### Runnable Lambda 

Changing callable to runnable 

In [ ]:
#simple function

from langchain_core.runnables import RunnableLambda


def make_upper(text):
    return text.upper()

#tunning callable into Runnable
chain=RunnableLambda(make_upper)

result=chain.invoke("Joe Harson")
print(result)

JOE HARSON


In [15]:
#chain multiple functions 

def add_exclamation(text):
    return text+"!!!"

chain=(

    RunnableLambda(make_upper) | #change lower to upper
    RunnableLambda(add_exclamation) #add exclamation
)

result=chain.invoke("Sandy")
print(result)

SANDY!!!


### RunnableParallel - runs multiple runnables at same time and merge results.

In [16]:
from langchain_core.runnables import RunnableParallel , RunnableLambda

def count_words(text):
    return len(text.split(" "))
def count_char (text):
    return len(text)
def upper_word(text):
    return text.upper()

parallel_chain=RunnableParallel(
    words= RunnableLambda(count_words),
    chars=RunnableLambda(count_char),
    upper=RunnableLambda(upper_word)
)

result=parallel_chain.invoke("I Love Chicken")
print(result)

{'words': 3, 'chars': 14, 'upper': 'I LOVE CHICKEN'}


trying chain in Runnable parallel

In [17]:
def last(text):
    return text[-1]


parallel_chain=RunnableParallel(
    words= RunnableLambda(count_words),
    chars=RunnableLambda(count_char),
    upper=RunnableLambda(upper_word) | RunnableLambda(last)
)

result=parallel_chain.invoke("I Love Chicken")
print(result)

{'words': 3, 'chars': 14, 'upper': 'N'}


### Merging all - RunnablePassthrough & RunnableLambda & RunnableParallel 

In [24]:
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough

def count(text):
    return len(text.split(" "))
def uppers(text):
    return text.upper()

chain=RunnablePassthrough.assign(
    analysis=RunnableParallel(
        Count=RunnableLambda(lambda x: count(x["text"])),
        upper =RunnableLambda(lambda x:uppers(x["text"]))

    )
)

result=chain.invoke({"text":"I am Coding in Python"})
print(result)

{'text': 'I am Coding in Python', 'analysis': {'Count': 5, 'upper': 'I AM CODING IN PYTHON'}}


While giving .aggign() we need to give input as  Dictionart , otherwise striing  is applicable